# Synthetic Data Generation for Evaluation

This notebook teaches a structured approach to generating diverse, realistic synthetic data for evaluating AI systems.

**The Problem:** Asking LLMs to generate queries without structure produces repetitive, generic outputs.

**The Solution:** Dimension-based generation with separated tuple creation and natural language phrasing.

In [ ]:
!pip install groq -q

In [ ]:
import getpass
import os
import json
import random
from groq import Groq

os.environ["GROQ_API_KEY"] = getpass.getpass("Enter your GROQ API key: ")
client = Groq()

---
## The Problem: Unstructured Generation

Let's see what happens when we ask an LLM to generate queries without structure.

In [ ]:
# Naive approach: Ask for queries directly
naive_prompt = """Generate 10 product search queries that a user might type into an e-commerce website.

Just list the queries, one per line."""

response = client.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": naive_prompt}],
    temperature=0.7
)

print("NAIVE APPROACH - Generated Queries:")
print("=" * 50)
print(response.choices[0].message.content)

### Problems with Naive Approach

Notice the issues:
- **Repetitive patterns**: Similar structure and phrasing
- **Limited diversity**: Missing edge cases and variations
- **Generic**: Doesn't capture real user behavior variety
- **No coverage guarantee**: Important scenarios may be missed

---
## Step 1: Define Dimensions

Identify the key dimensions that capture user behavior variations.

In [ ]:
# Define dimensions for product search queries
dimensions = {
    "category": [
        "Electronics", "Clothing", "Home & Kitchen", 
        "Sports", "Books", "Beauty", "Toys"
    ],
    "price_intent": [
        "budget",      # "cheap", "under $20"
        "mid-range",   # no price mention
        "premium",     # "best", "high-end", "premium"
        "deal-seeking" # "sale", "discount", "offer"
    ],
    "specificity": [
        "vague",       # "something for kitchen"
        "moderate",    # "wireless headphones"
        "specific",    # "Sony WH-1000XM5"
        "comparative"  # "better than X", "vs"
    ],
    "user_context": [
        "gift",        # "gift for mom"
        "personal",    # "for myself"
        "replacement", # "replacement for broken X"
        "first-time"   # "beginner", "starter"
    ],
    "urgency": [
        "none",        # no time pressure
        "fast-shipping", # "next day", "quick delivery"
        "seasonal"     # "Christmas", "back to school"
    ]
}

print("Dimensions defined:")
for dim, values in dimensions.items():
    print(f"  {dim}: {len(values)} values")
print(f"\nTotal possible combinations: {eval('*'.join(str(len(v)) for v in dimensions.values())):,}")

---
## Step 2: Identify Failure Modes

Target scenarios where the system is likely to fail.

In [ ]:
# Common failure modes for product search
failure_modes = [
    "misspellings",           # "headhpones", "laptpo"
    "abbreviations",          # "tv", "ac", "hp laptop"
    "colloquial_terms",       # "comfy shoes", "fancy dress"
    "negations",              # "not wireless", "without bluetooth"
    "multi_intent",           # "laptop for gaming and work"
    "implicit_constraints",   # "for small apartment" (implies compact)
    "brand_confusion",        # "like Apple but Android"
    "regional_terms"          # "trainers" vs "sneakers"
]

print("Failure modes to cover:")
for mode in failure_modes:
    print(f"  - {mode}")

---
## Step 3: Create Manual Tuples

Manually create ~20 tuples combining dimension values. This ensures coverage of important combinations.

In [ ]:
# Manually crafted tuples (category, price_intent, specificity, user_context, urgency, failure_mode)
manual_tuples = [
    # Standard cases
    ("Electronics", "premium", "specific", "personal", "none", None),
    ("Clothing", "budget", "vague", "gift", "seasonal", None),
    ("Home & Kitchen", "mid-range", "moderate", "replacement", "fast-shipping", None),
    ("Sports", "deal-seeking", "comparative", "first-time", "none", None),
    ("Books", "budget", "specific", "gift", "fast-shipping", None),
    
    # Edge cases with failure modes
    ("Electronics", "budget", "moderate", "personal", "none", "misspellings"),
    ("Clothing", "premium", "vague", "gift", "none", "colloquial_terms"),
    ("Home & Kitchen", "mid-range", "moderate", "personal", "none", "implicit_constraints"),
    ("Electronics", "premium", "comparative", "personal", "none", "brand_confusion"),
    ("Sports", "budget", "moderate", "first-time", "none", "abbreviations"),
    
    # More diverse combinations
    ("Beauty", "premium", "specific", "gift", "seasonal", None),
    ("Toys", "mid-range", "vague", "gift", "fast-shipping", None),
    ("Electronics", "deal-seeking", "moderate", "replacement", "none", None),
    ("Clothing", "mid-range", "specific", "personal", "none", "regional_terms"),
    ("Home & Kitchen", "budget", "vague", "first-time", "none", "multi_intent"),
    
    # Challenging combinations
    ("Electronics", "budget", "comparative", "gift", "fast-shipping", "negations"),
    ("Sports", "premium", "specific", "personal", "seasonal", None),
    ("Beauty", "deal-seeking", "vague", "personal", "none", "colloquial_terms"),
    ("Books", "mid-range", "moderate", "gift", "none", "abbreviations"),
    ("Toys", "budget", "moderate", "gift", "seasonal", "misspellings"),
]

print(f"Created {len(manual_tuples)} manual tuples")
print("\nSample tuples:")
for t in manual_tuples[:5]:
    print(f"  {t}")

---
## Step 4a: Scale Tuple Generation with LLM

Use the LLM to generate more tuples following the same structure.

In [ ]:
def generate_more_tuples(existing_tuples, dimensions, n=10):
    """Generate additional tuples using LLM"""
    
    prompt = f"""Generate {n} new tuples for product search query generation.

Each tuple has these dimensions:
- category: {dimensions['category']}
- price_intent: {dimensions['price_intent']}
- specificity: {dimensions['specificity']}
- user_context: {dimensions['user_context']}
- urgency: {dimensions['urgency']}
- failure_mode: {failure_modes + [None]}

Example tuples:
{json.dumps(existing_tuples[:5], indent=2)}

Generate {n} NEW tuples that are different from the examples.
Focus on diverse, interesting combinations.
Return as a JSON array of arrays."""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.8
    )
    
    try:
        # Extract JSON from response
        text = response.choices[0].message.content
        start = text.find('[')
        end = text.rfind(']') + 1
        return json.loads(text[start:end])
    except:
        return []

# Generate more tuples
llm_tuples = generate_more_tuples(manual_tuples, dimensions, n=10)
print(f"Generated {len(llm_tuples)} additional tuples:")
for t in llm_tuples[:5]:
    print(f"  {t}")

---
## Step 4b: Convert Tuples to Natural Language

**Key Insight:** Separate tuple generation from query phrasing to avoid repetition.

In [ ]:
def tuple_to_query(tuple_data):
    """Convert a structured tuple to a natural language query"""
    
    category, price_intent, specificity, user_context, urgency, failure_mode = tuple_data
    
    prompt = f"""Convert this structured specification into a natural, realistic product search query.

Specification:
- Category: {category}
- Price Intent: {price_intent}
- Specificity: {specificity}
- User Context: {user_context}
- Urgency: {urgency}
- Failure Mode: {failure_mode if failure_mode else 'none'}

Guidelines:
- Write as a real user would type (casual, possibly incomplete)
- If failure_mode is 'misspellings', include a realistic typo
- If failure_mode is 'abbreviations', use common abbreviations
- If failure_mode is 'colloquial_terms', use informal language
- If failure_mode is 'negations', include what user doesn't want
- Match the specificity level (vague = few details, specific = exact product)
- Reflect the user context naturally

Return ONLY the search query, nothing else."""

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.9  # Higher temperature for variety
    )
    
    return response.choices[0].message.content.strip().strip('"')

# Convert sample tuples to queries
print("Converting tuples to natural language queries:")
print("=" * 60)

sample_queries = []
for i, t in enumerate(manual_tuples[:10]):
    query = tuple_to_query(t)
    sample_queries.append({"tuple": t, "query": query})
    print(f"\nTuple: {t}")
    print(f"Query: {query}")

---
## Complete Pipeline: Generate Diverse Dataset

In [ ]:
def generate_diverse_dataset(manual_tuples, n_total=20):
    """Generate a diverse dataset using the structured approach"""
    
    dataset = []
    
    # Use manual tuples first
    tuples_to_use = manual_tuples[:n_total]
    
    print(f"Generating {len(tuples_to_use)} queries...")
    
    for i, t in enumerate(tuples_to_use):
        query = tuple_to_query(t)
        dataset.append({
            "id": i + 1,
            "tuple": {
                "category": t[0],
                "price_intent": t[1],
                "specificity": t[2],
                "user_context": t[3],
                "urgency": t[4],
                "failure_mode": t[5]
            },
            "query": query
        })
        print(f"  [{i+1}/{len(tuples_to_use)}] {query[:50]}...")
    
    return dataset

# Generate the dataset
synthetic_dataset = generate_diverse_dataset(manual_tuples, n_total=15)
print(f"\nGenerated {len(synthetic_dataset)} queries")

---
## Compare: Naive vs Structured

In [ ]:
print("COMPARISON: Naive vs Structured Generation")
print("=" * 70)

print("\n📋 STRUCTURED APPROACH (Dimension-Based):")
print("-" * 50)
for item in synthetic_dataset[:10]:
    t = item['tuple']
    print(f"[{t['category'][:4]}, {t['price_intent'][:4]}, {t['specificity'][:4]}] → {item['query']}")

print("\n" + "=" * 70)
print("\n✅ Benefits of Structured Approach:")
print("  1. Guaranteed coverage of important dimensions")
print("  2. Explicit failure mode testing")
print("  3. Traceable - know WHY each query was generated")
print("  4. Reproducible - same tuples → similar coverage")
print("  5. Diverse - systematic variation across dimensions")

---
## Export Dataset

In [ ]:
# Export to JSON
output = {
    "metadata": {
        "dimensions": dimensions,
        "failure_modes": failure_modes,
        "total_queries": len(synthetic_dataset)
    },
    "queries": synthetic_dataset
}

with open("synthetic_queries.json", "w") as f:
    json.dump(output, f, indent=2)

print("Dataset exported to synthetic_queries.json")
print(f"\nDataset summary:")
print(f"  Total queries: {len(synthetic_dataset)}")
print(f"  With failure modes: {sum(1 for q in synthetic_dataset if q['tuple']['failure_mode'])}")
print(f"  Categories covered: {len(set(q['tuple']['category'] for q in synthetic_dataset))}")

---
## Summary

### The Structured Approach

```
1. Define Dimensions     →  What variations matter?
2. Identify Failures     →  What might break?
3. Create Manual Tuples  →  Ensure key coverage
4. Scale with LLM        →  Generate more tuples + Convert to natural language
```

### Key Principles

| Principle | Why It Matters |
|-----------|----------------|
| **Separate tuple gen from phrasing** | Avoids repetitive patterns |
| **Manual tuples first** | Guarantees coverage of critical cases |
| **Explicit failure modes** | Tests edge cases systematically |
| **High temperature for phrasing** | More natural variety |

### When to Use This Approach

- Evaluating search/retrieval systems
- Testing chatbots and customer support
- Building training data for classifiers
- Any scenario needing diverse, realistic user inputs